# 03 - Full-Resolution Inference, PQ Evaluation & Submission Pipeline

**Competition:** Solar Filament Segmentation Challenge 2026  
**Tasks:** Full-resolution tiled inference, threshold selection, instance extraction, diagnostic Panoptic Quality (PQ), and competition submission generation.

> **Principle:** Tiled inference with Hann window blending avoids patch-boundary seams. Instances are formatted as compressed COCO RLE counts per official competition specifications.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / 'src' / 'filament_seg' / 'submission.py').is_file() else cwd.parent
assert (project_root / 'src' / 'filament_seg' / 'submission.py').is_file(), cwd
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

from filament_seg.utils import load_config, get_device
from filament_seg.model import build_model
from filament_seg.train import load_checkpoint
from filament_seg.dataset import SolarFilamentDataset
from filament_seg.inference import predict_tiled, iter_probability_instances
from filament_seg.evaluate import diagnostic_panoptic_quality, evaluate_full_resolution_dataset
from filament_seg.submission import encode_coco_counts, decode_coco_counts, validate_submission_rows, generate_submission

device = get_device()
print(f'Python: {sys.executable}')
print(f'Device: {device}')

## 1. Load Trained Checkpoint

Load the best U-Net model (`best_model.pth`) without retraining.

In [ ]:
ckpt_path = project_root / 'checkpoints' / 'unet_resnet34' / 'best_model.pth'
model = build_model(pretrained=False).to(device)
load_checkpoint(ckpt_path, model, map_location=device)
model.eval()
print('Successfully loaded best baseline checkpoint.')

## 2. Full-Resolution Tiled Inference with Hann Window Blending

Demonstrate seamless 2048x2048 prediction with 512x512 tiles and 128px cosine overlap blending.

In [ ]:
test_dir = project_root / 'data' / 'filament-segmentation-2026' / 'MAGFiLO_1.0_Kaggle_2026' / 'test' / 'test_images'
test_images = sorted(test_dir.glob('*.jpeg'))
assert len(test_images) == 180, f'Expected 180 test images, found {len(test_images)}'

sample_test = test_images[0]
import cv2
img = cv2.imread(str(sample_test), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0

prob_map = predict_tiled(model, img, tile_size=512, overlap=128, device=device)
print(f'Test image: {sample_test.name}')
print(f'Output shape: {prob_map.shape}, range: [{prob_map.min():.4f}, {prob_map.max():.4f}]')

## 3. Thresholding, Instance Extraction & Panoptic Quality (PQ)

Extract individual filament instances using validated optimal parameters (threshold=0.75, min_area=300).

In [ ]:
train_dir = project_root / 'data' / 'filament-segmentation-2026' / 'MAGFiLO_1.0_Kaggle_2026' / 'train'
dataset = SolarFilamentDataset(train_dir / 'train_images', train_dir / 'MAGFiLO_1.0_Annotations_kaggle2026_train.json')
indices = np.random.default_rng(42).permutation(len(dataset))
val_indices = indices[565:].tolist()

# Evaluate on sample validation image
val_sample = dataset[val_indices[0]]
val_prob = predict_tiled(model, val_sample['image'], tile_size=512, overlap=128, device=device)
gt_instances = [inst['mask'] for inst in dataset.iter_instances(val_indices[0], 0)]
pred_instances = list(iter_probability_instances(val_prob, threshold=0.75, min_area=300))

pq_res = diagnostic_panoptic_quality(gt_instances, pred_instances)
print(f'Sample Validation File: {val_sample["filename"]}')
print(f'Ground Truth Instances: {len(gt_instances)}, Predicted Instances: {len(pred_instances)}')
print(f'Diagnostic PQ: {pq_res["pq"]:.4f} (TP={pq_res["tp"]}, FP={pq_res["fp"]}, FN={pq_res["fn"]})')

## 4. Compressed COCO RLE Verification & Schema Conformance

Verify compressed COCO string counts round-trip encoding and structural schema validation.

In [ ]:
if pred_instances:
    sample_mask = pred_instances[0]
    counts = encode_coco_counts(sample_mask)
    decoded = decode_coco_counts(counts, sample_mask.shape)
    assert np.array_equal(decoded, sample_mask), 'RLE round trip failed!'
    print('COCO RLE Round-Trip: PASS')
    print(f'Encoded counts snippet: {counts[:40]}... (length: {len(counts)})')

## 5. Generate Submission

Generate competition submission CSV across test images using validated postprocessing parameters.

In [ ]:
submission_csv_path = project_root / 'outputs' / 'submission.csv'
# Generate predictions for sample test set (or full test set)
df_submission = generate_submission(
    model=model,
    test_images=test_images[:5],  # demonstration on first 5 images
    output_csv_path=submission_csv_path,
    device=device,
    threshold=0.75,
    min_area=300,
    tile_size=512,
    overlap=128,
)
print('Submission generation completed.')
print('Submission preview:')
display(df_submission.head())
print(f'Total rows generated: {len(df_submission)}')